# Daily AI Radar LinkedIn Agent

This notebook contains the full manual agent flow. It does not save your Anthropic API key to GitHub or to a local `.env` file.

Run the cells from top to bottom. The final cell writes four markdown files to `outputs/`.

## 1. Install Notebook Dependencies

This cell installs packages into the current notebook kernel. If it says your Python version is unsupported, switch the notebook kernel to Python 3.11 or 3.12.

In [ ]:
import sys
import subprocess
from importlib.util import find_spec

print("Notebook Python:", sys.version)
if sys.version_info < (3, 10) or sys.version_info >= (3, 14):
    raise RuntimeError("CrewAI needs Python 3.10, 3.11, 3.12, or 3.13. Recommended: Python 3.11.")

required_packages = [
    "crewai>=0.80.0",
    "anthropic>=0.40.0",
    "python-dotenv>=1.0.1",
    "ddgs>=9.0.0",
    "feedparser>=6.0.11",
    "PyYAML>=6.0.2",
]

import_names = ["crewai", "anthropic", "dotenv", "ddgs", "feedparser", "yaml"]
missing = [name for name in import_names if find_spec(name) is None]

if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", *required_packages])
    print("Install complete. Restart the kernel, then run the notebook from the top.")
else:
    print("All required packages are already available in this kernel.")

## 2. Enter Runtime Settings

The key is stored only in notebook memory for this run.

In [ ]:
import os
from getpass import getpass

os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your Anthropic API key. It will not be saved: ").strip()
os.environ.setdefault("APP_TIMEZONE", "America/Chicago")
os.environ.setdefault("ANTHROPIC_MODEL", "claude-3-5-sonnet-latest")

if not os.environ["ANTHROPIC_API_KEY"]:
    raise RuntimeError("Anthropic API key is required to run the agent.")

print("Model:", os.environ["ANTHROPIC_MODEL"])
print("Timezone:", os.environ["APP_TIMEZONE"])

## 3. Imports, Paths, and Memory Files

In [ ]:
import csv
import json
import os
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime
from pathlib import Path
from typing import Any
from urllib.parse import quote_plus, urlparse
from zoneinfo import ZoneInfo

import feedparser
from crewai import Agent, Crew, LLM, Process, Task
from ddgs import DDGS

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "outputs"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def ensure_csv(path: Path, columns: list[str]) -> None:
    if path.exists():
        return
    with path.open("w", newline="", encoding="utf-8") as csvfile:
        csv.writer(csvfile).writerow(columns)

def ensure_text(path: Path, content: str) -> None:
    if not path.exists():
        path.write_text(content.strip() + "\n", encoding="utf-8")

ensure_csv(DATA_DIR / "past_posts.csv", ["date", "topic", "angle", "post_text", "visual_type", "notes"])
ensure_csv(DATA_DIR / "post_performance.csv", ["date", "topic", "angle", "views", "likes", "comments", "reposts", "profile_visits", "connection_requests", "recruiter_engagement", "executive_engagement", "notes"])
ensure_text(DATA_DIR / "positioning_rules.md", """
# Positioning Rules

Ayesha Saeed-Haq translates enterprise AI signals into executive decision intelligence.

Emphasize enterprise AI execution, decision intelligence, AI governance, CRM/NBA systems, experimentation, risk detection, Fortune-20 operating systems, and AI Radar positioning.

Avoid generic AI commentary, hype language, beginner education, unsupported statistics, and motivational filler.
""")
ensure_text(DATA_DIR / "ai_radar_themes.md", """
# AI Radar Themes

- AI execution vs hype
- Governance debt
- Decision intelligence
- Competitive intelligence
- Agentic operating models
- Workforce redesign
- AI maturity
- AI commercialization
- Boardroom implications
""")
print("Working folder:", ROOT)

## 4. Source Credibility and Signal Scoring

In [ ]:
HIGH_TRUST_DOMAINS = {
    "sec.gov", "beckershospitalreview.com", "modernhealthcare.com", "fiercehealthcare.com", "healthcaredive.com",
    "wsj.com", "ft.com", "fortune.com", "bloomberg.com", "cnbc.com", "mckinsey.com", "accenture.com",
    "deloitte.com", "bain.com", "bcg.com", "venturebeat.com", "theinformation.com", "techcrunch.com",
    "aws.amazon.com", "microsoft.com", "googlecloud.com", "openai.com", "anthropic.com",
}
LOW_TRUST_HINTS = ["blogspot", "substack", "medium.com", "newsletter", "seo", "top10", "best-ai-tools"]

def domain_from_url(url: str) -> str:
    return urlparse(url).netloc.lower().removeprefix("www.")

def credibility_score(url: str, source_name: str = "") -> tuple[int, str]:
    domain = domain_from_url(url)
    source_text = f"{domain} {source_name}".lower()
    if any(domain == trusted or domain.endswith(f".{trusted}") for trusted in HIGH_TRUST_DOMAINS):
        return 90, "High-trust source or named whitelist domain."
    if any(hint in source_text for hint in LOW_TRUST_HINTS):
        return 35, "Lower trust: social, newsletter, SEO, or repost-style source."
    if any(term in source_text for term in ["investor", "press", "newsroom", "ir."]):
        return 85, "Likely primary company source."
    return 60, "Standard source; verify before making strong claims."

WEIGHTS = {
    "executive_relevance": 25,
    "hiring_manager_relevance": 20,
    "ai_radar_alignment": 20,
    "freshness": 10,
    "tension": 10,
    "visual_potential": 10,
    "credibility": 5,
}
EXECUTIVE_TERMS = ["enterprise", "board", "governance", "operating model", "risk", "roi", "strategy", "decision", "commercial"]
HIRING_MANAGER_TERMS = ["execution", "platform", "analytics", "automation", "workforce", "crm", "experimentation", "infrastructure"]
AI_RADAR_TERMS = ["ai", "agent", "governance", "decision intelligence", "commercialization", "competitive", "capex", "lawsuit", "partnership"]
TENSION_TERMS = ["lawsuit", "layoff", "risk", "cost", "capex", "trust", "regulation", "margin", "accountability"]

def signal_text(signal: dict[str, Any]) -> str:
    return " ".join(str(signal.get(key, "")) for key in ["title", "summary", "sector", "company"]).lower()

def term_score(text: str, terms: list[str]) -> float:
    return min(1.0, sum(1 for term in terms if term in text) / 4)

def freshness_score(date_value: str | None) -> float:
    if not date_value:
        return 0.4
    try:
        parsed = parsedate_to_datetime(date_value)
    except Exception:
        try:
            parsed = datetime.fromisoformat(date_value.replace("Z", "+00:00"))
        except Exception:
            return 0.4
    if parsed.tzinfo is None:
        parsed = parsed.replace(tzinfo=timezone.utc)
    age_days = max(0, (datetime.now(timezone.utc) - parsed).days)
    if age_days <= 1:
        return 1.0
    if age_days <= 3:
        return 0.8
    if age_days <= 7:
        return 0.6
    if age_days <= 14:
        return 0.4
    return 0.2

def score_signal(signal: dict[str, Any]) -> dict[str, Any]:
    text = signal_text(signal)
    components = {
        "executive_relevance": term_score(text, EXECUTIVE_TERMS),
        "hiring_manager_relevance": term_score(text, HIRING_MANAGER_TERMS),
        "ai_radar_alignment": term_score(text, AI_RADAR_TERMS),
        "freshness": freshness_score(signal.get("publication_date")),
        "tension": term_score(text, TENSION_TERMS),
        "visual_potential": 1.0 if any(term in text for term in ["map", "trend", "framework", "platform", "operating model", "capex"]) else 0.6,
        "credibility": float(signal.get("credibility_score", 50)) / 100,
    }
    signal["component_scores"] = components
    signal["score"] = round(sum(components[key] * WEIGHTS[key] for key in WEIGHTS))
    return signal

def score_signals(signals: list[dict[str, Any]]) -> list[dict[str, Any]]:
    return sorted([score_signal(signal) for signal in signals], key=lambda item: item["score"], reverse=True)

## 5. Search DDGS and Google News RSS

In [ ]:
SEARCH_CATEGORIES = [
    "healthcare AI enterprise governance",
    "telecom AI automation workforce",
    "fintech AI risk decision systems",
    "enterprise AI execution Fortune 100",
    "AI governance enterprise lawsuits",
    "AI layoffs workforce redesign",
    "AI capex cloud infrastructure",
    "AI partnerships commercialization",
    "agentic AI operating systems enterprise",
    "boardroom AI strategy decision intelligence",
]
RSS_QUERIES = ["enterprise AI governance", "AI workforce redesign", "AI capex enterprise", "AI decision intelligence", "agentic AI enterprise"]

def search_ddgs(max_results: int = 20) -> list[dict[str, Any]]:
    signals = []
    seen_urls = set()
    with DDGS() as ddgs:
        for query in SEARCH_CATEGORIES:
            for result in ddgs.text(query, max_results=3, timelimit="d"):
                url = result.get("href") or result.get("url") or ""
                if not url or url in seen_urls:
                    continue
                seen_urls.add(url)
                score, note = credibility_score(url, result.get("source", ""))
                signals.append({
                    "title": result.get("title", "Untitled signal"),
                    "company": "Unknown",
                    "sector": query,
                    "source": result.get("source", "DDGS"),
                    "url": url,
                    "publication_date": result.get("date", ""),
                    "summary": result.get("body", ""),
                    "executive_relevance": "Potential enterprise AI decision, governance, workforce, or commercialization signal.",
                    "credibility_score": score,
                    "credibility_note": note,
                    "why_it_matters": "Needs executive framing against operating model, governance, ROI, and decision infrastructure.",
                })
                if len(signals) >= max_results:
                    return signals
    return signals

def read_google_news(max_results: int = 20) -> list[dict[str, Any]]:
    signals = []
    seen_urls = set()
    for query in RSS_QUERIES:
        url = f"https://news.google.com/rss/search?q={quote_plus(query)}&hl=en-US&gl=US&ceid=US:en"
        feed = feedparser.parse(url)
        for entry in feed.entries[:5]:
            link = entry.get("link", "")
            if not link or link in seen_urls:
                continue
            seen_urls.add(link)
            score, note = credibility_score(link, entry.get("source", {}).get("title", ""))
            signals.append({
                "title": entry.get("title", "Untitled Google News signal"),
                "company": "Unknown",
                "sector": query,
                "source": entry.get("source", {}).get("title", "Google News RSS"),
                "url": link,
                "publication_date": entry.get("published", ""),
                "summary": entry.get("summary", ""),
                "executive_relevance": "Fresh market signal for executive AI Radar review.",
                "credibility_score": score,
                "credibility_note": note,
                "why_it_matters": "May indicate a shift in enterprise AI execution, governance, spend, or workforce design.",
            })
            if len(signals) >= max_results:
                return signals
    return signals

def collect_signals() -> list[dict[str, Any]]:
    signals = []
    errors = []
    for label, loader in [("DDGS", search_ddgs), ("Google News RSS", read_google_news)]:
        try:
            signals.extend(loader())
        except Exception as exc:
            errors.append(f"{label} failed: {exc}")
    scored = score_signals(signals)
    if errors:
        scored.append({"title": "Collection warning", "company": "System", "sector": "Operations", "source": "Local runtime", "url": "", "publication_date": "", "summary": " | ".join(errors), "executive_relevance": "Search failure should be investigated before publishing.", "credibility_score": 0, "credibility_note": "Runtime warning", "why_it_matters": "The output may be incomplete if a source layer failed.", "score": 0, "component_scores": {}})
    return scored[:10]

signals = collect_signals()
len(signals), signals[:2]

## 6. CrewAI Agents and Tasks

In [ ]:
def today_slug(timezone_name: str = "America/Chicago") -> str:
    return datetime.now(ZoneInfo(timezone_name)).strftime("%Y-%m-%d")

def now_human(timezone_name: str = "America/Chicago") -> str:
    current = datetime.now(ZoneInfo(timezone_name))
    return f"{current.strftime('%B %d, %Y at')} {current.strftime('%I:%M %p %Z').lstrip('0')}"

def signals_markdown(signals: list[dict[str, Any]]) -> str:
    blocks = []
    for index, signal in enumerate(signals, start=1):
        blocks.append("\n".join([
            f"## {index}. {signal.get('title')}",
            f"- Score: {signal.get('score')}",
            f"- Company: {signal.get('company')}",
            f"- Sector: {signal.get('sector')}",
            f"- Source: {signal.get('source')}",
            f"- URL: {signal.get('url')}",
            f"- Publication date: {signal.get('publication_date')}",
            f"- Summary: {signal.get('summary')}",
            f"- Executive relevance: {signal.get('executive_relevance')}",
            f"- Credibility: {signal.get('credibility_score')} - {signal.get('credibility_note')}",
            f"- Why it matters: {signal.get('why_it_matters')}",
        ]))
    return "\n\n".join(blocks)

def write_markdown(path: Path, content: str) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content.strip() + "\n", encoding="utf-8")
    return path

def build_llm() -> LLM:
    return LLM(model=f"anthropic/{os.environ['ANTHROPIC_MODEL']}", api_key=os.environ["ANTHROPIC_API_KEY"])

def build_agents(llm: LLM) -> dict[str, Agent]:
    return {
        "signal_scout": Agent(role="Signal Scout", goal="Find and explain high-signal enterprise AI developments without chasing hype.", backstory="You are an enterprise AI intelligence analyst. You separate durable executive signals from noisy AI commentary.", llm=llm, verbose=True),
        "strategist": Agent(role="Executive Strategist", goal="Select the strongest AI Radar story and differentiated executive framing.", backstory="You advise enterprise AI leaders through decision intelligence, governance debt, workforce redesign, commercialization, and boardroom consequences.", llm=llm, verbose=True),
        "post_writer": Agent(role="LinkedIn Post Writer", goal="Write crisp executive LinkedIn posts that create recruiter and VP-level attention.", backstory="You write in Ayesha Saeed-Haq's executive voice: direct, analytical, evidence-based, short paragraphs, no hashtags, no emojis, no generic AI hype.", llm=llm, verbose=True),
        "qc_agent": Agent(role="Hallucination and Credibility QC Agent", goal="Protect credibility by removing unsupported claims, weak framing, generic language, and fake precision.", backstory="You are a rigorous executive editor. You rewrite anything that sounds fabricated, inflated, generic, repetitive, or unclear about business consequence.", llm=llm, verbose=True),
    }

def build_tasks(agents: dict[str, Agent], signals_md: str, memory_context: str) -> list[Task]:
    signal_review = Task(description=f"Review grounded signals collected for {now_human(os.environ['APP_TIMEZONE'])}.\n\n{signals_md}\n\nReturn top 10 ranked signals. Do not invent facts.", expected_output="Ranked top 10 enterprise AI signals with source URLs, credibility notes, and executive relevance.", agent=agents["signal_scout"])
    strategy = Task(description=f"Choose the single strongest story and framing for Ayesha Saeed-Haq.\n\nPositioning context:\n{memory_context}\n\nOptimize for VP relevance, recruiter attraction, enterprise credibility, repeat audience growth, and profile visits.", expected_output="Selected Topic, Selected Angle, Why Today, Confidence Score, and selection reasoning.", agent=agents["strategist"], context=[signal_review])
    post = Task(description="Write one ready-to-publish LinkedIn post. Requirements: strong hook, short paragraphs, one business tension, enterprise consequence, practical implication, decision consequence, no hashtags, no emojis, no clickbait, no unsupported statistics, no fake numbers.", expected_output="One final LinkedIn post, ready to publish.", agent=agents["post_writer"], context=[signal_review, strategy])
    qc = Task(description="Quality-control the post and package final outputs. Rewrite if it has unsupported claims, weak hook, generic language, fake precision, repetitive framing, unclear consequence, or low executive value. Return valid JSON with exactly these keys: daily_post, visual_brief, sources, reasoning_log.", expected_output="Valid JSON object with daily_post, visual_brief, sources, and reasoning_log markdown strings.", agent=agents["qc_agent"], context=[signal_review, strategy, post])
    return [signal_review, strategy, post, qc]

## 7. Run the Agent and Save Outputs

In [ ]:
signals_md = signals_markdown(signals)
memory_context = "\n\n".join([
    (DATA_DIR / "positioning_rules.md").read_text(encoding="utf-8"),
    (DATA_DIR / "ai_radar_themes.md").read_text(encoding="utf-8"),
])

llm = build_llm()
agents = build_agents(llm)
tasks = build_tasks(agents, signals_md, memory_context)
crew = Crew(agents=list(agents.values()), tasks=tasks, process=Process.sequential, verbose=True)
result = await crew.kickoff_async()

raw_text = str(result).strip()
if raw_text.startswith("```"):
    raw_text = raw_text.strip("`").removeprefix("json").strip()

try:
    documents = json.loads(raw_text)
except Exception:
    documents = {
        "daily_post": raw_text,
        "visual_brief": "# Visual Brief\n\nReview the final post and create an AI Radar visual using yellow, black, and white branding.",
        "sources": f"# Sources\n\n{signals_md}",
        "reasoning_log": f"# Reasoning Log\n\nCrew output was not valid JSON. Raw output preserved in daily_post.\n\n{signals_md}",
    }

date_slug = today_slug(os.environ["APP_TIMEZONE"])
output_paths = {
    "daily_post": write_markdown(OUTPUT_DIR / f"{date_slug}_daily_post.md", documents["daily_post"]),
    "visual_brief": write_markdown(OUTPUT_DIR / f"{date_slug}_visual_brief.md", documents["visual_brief"]),
    "sources": write_markdown(OUTPUT_DIR / f"{date_slug}_sources.md", documents["sources"]),
    "reasoning_log": write_markdown(OUTPUT_DIR / f"{date_slug}_reasoning_log.md", documents["reasoning_log"]),
}
output_paths